In [1]:
import pandas as pd
import pickle
import importlib
#import ipynb
import sys
import os
from syllabification import tokenize, syllabify_sentences

In [2]:
#Test 
sentence = "bonjour comment allez vous"
tok_sent = tokenize(sentence)
print(tok_sent)
syll_sent = syllabify_sentences(tok_sent)
print(syll_sent)

['bonjour', 'comment', 'allez', 'vous']
['b§', 'ZuR', 'ko', 'm@', 'a', 'le', 'vu']
['b§', 'ZuR', 'ko', 'm@', 'a', 'le', 'vu']


In [ ]:
# Read each line as a sentence
tokenized_sentences = []
transcribed_sentences = []


path = "Z:/data/French/french_sentences.txt"  # Use the mounted drive letter

with open(path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        sentence = line.strip()
        if sentence:
            # Tokenize
            tokenized_sentence = tokenize(sentence)
            tokenized_sentences.append(tokenized_sentence)
            
            # Syllabify
            transcribed_sentence = syllabify_sentences(tokenized_sentence, language="French")
            print(transcribed_sentence)
            if transcribed_sentence: 
                transcribed_sentences.append(transcribed_sentence)

        # Optional: limit for testing
            if i >= 20:
             break


# show the entries 
for i, sentence in enumerate(transcribed_sentences[:20]):
    print(f"Sentence {i+1}: {sentence}")


# Save to .pkl
with open("produced_data/tokenized_sentences.pkl", "wb") as f:
    pickle.dump(tokenized_sentences, f)

with open("produced_data/transcribed_sentences.pkl", "wb") as f:
    pickle.dump(transcribed_sentences, f)

print("✅ Saved tokenized and transcribed sentences for this language to 'data/'")


['l°', 'Rwa', 'jom', 'd°', 'france', 'e', 's°', 'l8i', 'de', 'sj2']
['l°', 'Rwa', 'jom', 'd°', 'france', 'e', 's°', 'l8i', 'de', 'sj2']
['gER', 'd°', 'R°', 'li', 'Zj§', '§', 'fa', 'so', 'ne', 'd@', 'la', 'kEl', 'nu', 'vi', 'v§']
['gER', 'd°', 'R°', 'li', 'Zj§', '§', 'fa', 'so', 'ne', 'd@', 'la', 'kEl', 'nu', 'vi', 'v§']
['E', 'l°', 'k9R', 'e', 'Es', 'to', 'ma', '1', 'Rwa']
['E', 'l°', 'k9R', 'e', 'Es', 'to', 'ma', '1', 'Rwa']
['xvie', 'sjEkl', 'R°', 'li', 'Zj§', 'e', 'pu', 'vwaR', 's§', 'e', 'tRwa', 't°', 'm@', 'me', 'le']
['xvie', 'sjEkl', 'R°', 'li', 'Zj§', 'e', 'pu', 'vwaR', 's§', 'e', 'tRwa', 't°', 'm@', 'me', 'le']
['RiR']
['RiR']
['le', 'ka', 'to', 'lik', 'vu', 'de', 'zo', 'be', 'is']
['le', 'ka', 'to', 'lik', 'vu', 'de', 'zo', 'be', 'is']
['Z°', 'e', 'pu', 'z°', 'RE', 'sEt', 'e', 'Re', 'tik', 'vu', '@', 't@', 'de']
['Z°', 'e', 'pu', 'z°', 'RE', 'sEt', 'e', 'Re', 'tik', 'vu', '@', 't@', 'de']
['Za', 'mE']
['Za', 'mE']
['a', 've', 'fE', 'henri']
['a', 've', 'fE', 'henri']
[]
[]
['

In [4]:
from collections import defaultdict
from math import log2

def build_markov_chain(merged_sentences, n):
    """
    Build an n-gram transition probability model.
    """
    ngram_counts = defaultdict(lambda: defaultdict(int)) # maps prefix to next token e.g. {('I', 'am'): {'happy': 3, 'tired': 2}}
    
    # Create n-grams
    ngram_list = generate_ngrams(merged_sentences, n)

    # Count occurrences of each n-gram
    for gram in ngram_list:
        prefix, next_token = tuple(gram[:-1]), gram[-1]
        ngram_counts[prefix][next_token] += 1
    
    # Convert counts to probabilities
    transition_probs = defaultdict(dict)
    entropy = defaultdict(float)

    for prefix, suffix_counts in ngram_counts.items():
        prefix_occurences = sum(suffix_counts.values()) # total ocurrences of the prefix across all possible suffixes
        for suffix, suffix_count in suffix_counts.items():
            # Compute the conditional probability of the next syllable given the previous ones
            transition_probs[prefix][suffix] = suffix_count / prefix_occurences  # e.g. {('I', 'am'): {'happy': 0.6, 'tired': 0.4}}


    for prefix, suffix_probs in transition_probs.items():
        entropy[prefix] = -sum(p * log2(p) for p in suffix_probs.values() if p > 0)  # H(X | Y) e.g. {('I', 'am'): 0.97095}

    return transition_probs, entropy


Example: 

prefix = ('I', 'am')
suffix_counts = {'happy': 1, 'tired': 1} 
prefix_occurences = 1 + 1 = 2

transition_probs[('I', 'am')] = {
'happy': 1/2 = 0.5,
'tired': 1/2 = 0.5
}

In [ ]:
from nltk.util import ngrams

def generate_ngrams(transcribed_sentence, n):
    """
    Generate n-grams from a list of syllables.
    """
    return list(ngrams(transcribed_sentence, n, pad_left=True, pad_right=True, left_pad_symbol="<BOS>", right_pad_symbol="<EOS>"))

In [ ]:
with open("produced_data/transcribed_sentences.pkl", "rb") as f: 
    transcribed_sentences = pickle.load(f)
    
merged_sentences = []
for sentence in transcribed_sentences:
    merged_sentences.extend(sentence)  # Merge all sentences

n = 5
markov_models = {}  # Dictionary to store models for different n-gram sizes

for i in range(2,n):  # Create models for 2-grams, 3-grams and 4-grams
    markov_model[i] = build_markov_chain(merged_sentences, i)

In [ ]:
# TODO: Store frequency counts of each n-gram

In [ ]:
# TODO: Save learned transition probabilities 